<a href="https://colab.research.google.com/github/ivanjob64/Nair_Proyecto/blob/dev/Modelos_de_Regresi%C3%B3n_VFinal_V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importar las librerias necesarias
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns

%matplotlib inline

In [2]:
mpl.style.use(['ggplot'])

In [3]:
source_file="servicios.csv"
# Creamos un dataframe a partir del archivo csv descargado
df_servicios2 = pd.read_csv(source_file, sep=';',  encoding='latin-1')
tamano_bytes = os.path.getsize(source_file)
#tamaño de archivo
tamano_mb = tamano_bytes / (1024 * 1024)
print(f"📁 Tamaño del archivo: {tamano_mb:.2f} MB ({tamano_bytes:,} bytes)")
df_servicios2.shape

📁 Tamaño del archivo: 626.71 MB (657,149,228 bytes)


(5590161, 10)

In [4]:
# Estandarización de datos: Asegurar que los campos de tipo fecha son campos de tipo datetime
df_servicios2['FechaHoraConsumo'] = pd.to_datetime(df_servicios2['FechaHoraConsumo'], errors='coerce')
df_servicios2['FechaHoraRespuesta'] = pd.to_datetime(df_servicios2['FechaHoraRespuesta'], errors='coerce')
df_servicios2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5590161 entries, 0 to 5590160
Data columns (total 10 columns):
 #   Column              Dtype         
---  ------              -----         
 0   ï»¿id               int64         
 1   codCliente          object        
 2   canal               int64         
 3   empresaId           int64         
 4   empresa             object        
 5   estado              int64         
 6   intentos            int64         
 7   respuesta           object        
 8   FechaHoraConsumo    datetime64[ns]
 9   FechaHoraRespuesta  datetime64[ns]
dtypes: datetime64[ns](2), int64(5), object(3)
memory usage: 426.5+ MB


In [5]:
df_servicios2.rename(columns={'ï»¿id': 'id'}, inplace=True)
df_servicios=df_servicios2[(df_servicios2['FechaHoraConsumo'] < '2025-10-01')]
df_servicios.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5518946 entries, 0 to 5590160
Data columns (total 10 columns):
 #   Column              Dtype         
---  ------              -----         
 0   id                  int64         
 1   codCliente          object        
 2   canal               int64         
 3   empresaId           int64         
 4   empresa             object        
 5   estado              int64         
 6   intentos            int64         
 7   respuesta           object        
 8   FechaHoraConsumo    datetime64[ns]
 9   FechaHoraRespuesta  datetime64[ns]
dtypes: datetime64[ns](2), int64(5), object(3)
memory usage: 463.2+ MB


In [7]:
df_servicios['empresaId']=df_servicios['empresaId']+10000
df_servicios['codCliente'] = df_servicios['codCliente'].astype(str).apply(lambda x: x[:3] + '*' * (len(x) - 3))

/tmp/ipykernel_26078/3808640164.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_servicios['empresaId']=df_servicios['empresaId']+10000
/tmp/ipykernel_26078/3808640164.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_servicios['codCliente'] = df_servicios['codCliente'].astype(str).apply(lambda x: x[:3] + '*' * (len(x) - 3))


In [ ]:
df_servicios.head(5)

**Preparación de Datos:**

In [ ]:
#lista registros nulos
df_servicios[df_servicios.isnull().any(axis=1)]

In [ ]:
#cantidad de registros nulos
df_servicios.isna().sum()
#llena campos vacíos por la media
#df_servicios.fillna(df_servicios.mean())

In [ ]:
#lista registros duplicados
df_servicios.duplicated().sum()
#elimina registros duplicados
#f_servicios.dropduplicated()

In [ ]:
#elimina registros nulos
df_servicios = df_servicios.dropna()

In [ ]:
#listar valores únicos de campo intentos
df_servicios['intentos'].unique()

In [ ]:
#completar con valor 0 campo codCliente que contienen valores nulos  ***NO
#df_servicios['codCliente'] = df_servicios['codCliente'].fillna(0)
#df_servicios[df_servicios.isnull().any(axis=1)]

In [ ]:
fecha_min = df_servicios['FechaHoraConsumo'].min()
fecha_max = df_servicios['FechaHoraConsumo'].max()

print("Fecha mínima:", fecha_min)
print("Fecha máxima:", fecha_max)

In [ ]:
# listar empresas y cantidad de pagos
conteo = df_servicios['empresa'].value_counts().reset_index()
conteo.columns = ['empresa', 'frecuencia']
print(conteo)

In [ ]:
# listar canales y cantidad de pagos
conteo = df_servicios['canal'].value_counts().reset_index()
conteo.columns = ['canal', 'frecuencia']
print(conteo)

In [ ]:
#Crear columnas de periodo: año-mes, día y hora
df_servicios['FechaHoraConsumo'] = pd.to_datetime(df_servicios['FechaHoraConsumo'])
df_servicios['mes'] = df_servicios['FechaHoraConsumo'].dt.to_period('M')
df_servicios['dia_promedio'] = df_servicios['FechaHoraConsumo'].dt.dayofweek
df_servicios['hora_promedio'] = df_servicios['FechaHoraConsumo'].dt.hour



In [ ]:
df_servicios.info()

In [ ]:
pagos_mes = df_servicios.groupby(['empresa', 'mes']).agg(
    total_pagos=('id', 'count'),
    #exitos=('estado', lambda x: (x == 1).sum()),
    #errores=('estado', lambda x: (x != 1).sum()),
    tasa_exitos=('estado', lambda x: (x == 1).mean()),
    #tasa_errores=('estado', lambda x: (x != 1).mean()),
    promedio_intentos=('intentos', 'mean'),
    hora_promedio=('hora_promedio', 'mean'),
    dia_promedio=('dia_promedio', 'mean'),
).reset_index()

In [ ]:
#Calcular incremento mensual por empresa
pagos_mes = pagos_mes.sort_values(by=['empresa', 'mes'])
pagos_mes['incremento'] = pagos_mes.groupby('empresa')['total_pagos'].diff().fillna(0)

#calculo de variable objetivo (incremento) de pagos con respecto al anterior mes
pagos_mes['incremento'] = pagos_mes.groupby('empresa')['total_pagos'].diff()
#Reemplazar valores nulos (primer mes sin comparación)
pagos_mes['incremento'] = pagos_mes['incremento'].fillna(0)


In [ ]:
# Agregar columna para indicar tendencia
pagos_mes['tendencia'] = pagos_mes['incremento'].apply(
    lambda x: 'Incremento'
    if x > 0 else ('Disminución' if x < 0 else 'Sin cambio')
)


In [ ]:
pagos_mes['incremento'] = pagos_mes['incremento'].astype('int64')
pagos_mes['mes_num'] = pagos_mes['mes'].dt.month
pagos_mes['var_pagos'] = pagos_mes['total_pagos'].pct_change().fillna(0)

In [ ]:
empresas = pagos_mes['empresa'].unique()
print (empresas)

In [ ]:
pagos_mes.info()

In [ ]:
#agregar campo tendencia_num de tipo numérico
mapa_tendencia = {'Disminución': -1, 'Sin cambio': 0, 'Incremento': 1}
pagos_mes['tendencia_num'] = pagos_mes['tendencia'].map(mapa_tendencia)
pagos_mes.info()

In [ ]:
pagos_mes.head(5)

In [ ]:
#permitirá detectar patrones específicos por época del año
pagos_mes['mes_num'] = pagos_mes['mes'].dt.month
pagos_mes.info()

In [ ]:
#Crear variable var_pagos que calcula la variación porcentual de pagos entre registros consecutivos
pagos_mes['var_pagos'] = pagos_mes['total_pagos'].pct_change().fillna(0)
pagos_mes.info()

In [ ]:
pagos_mes.isna().sum()

In [ ]:
#calcular matriz de correlación
corr_pagos = pagos_mes.corr(numeric_only=True)
corr_pagos

In [ ]:
#matriz de correlación
sns.heatmap(corr_pagos, annot=True, cmap="RdBu", fmt=".2f")

In [ ]:
#definir variable objetivo y prodictorias
y = pagos_mes['incremento']
X = pagos_mes[['total_pagos', 'tasa_exitos','promedio_intentos','tendencia_num','mes_num','var_pagos','hora_promedio','dia_promedio']]



In [ ]:
#ver coheficientes del modelo
print("✅ Dataset enriquecido con éxito")
print("Dimensiones de X:", X.shape)
print("Variables predictoras:", list(X.columns))

In [ ]:
#dividir datos de prueba y entrenamiento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
#mostrar cantidad de datos entrenados

print(f" X_train → {X_train.shape[0]} registros y {X_train.shape[1]} columnas")
print(f" X_test  → {X_test.shape[0]} registros y {X_test.shape[1]} columnas")
print(f" y_train → {y_train.shape[0]} registros y 1 columna (variable objetivo)")
print(f" y_test  → {y_test.shape[0]} registros y 1 columna (variable objetivo)")

**PRIMER MODELO: REGRESIÓN LINEAL MÚLTIPLE**

In [ ]:
#NORMALIZACIÓN
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Crear modelo
modelo = LinearRegression()

# Entrenar modelo con datos de entrenamiento
modelo.fit(X_train, y_train)

#ver coheficientes del modelo
print("Intercepto (β₀):", modelo.intercept_)
print("Coeficientes (β):", modelo.coef_)


In [ ]:
# Realizar predicciones sobre el conjunto de prueba
y_pred = modelo.predict(X_test)
# Calcular métricas
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Mostrar resultados
print("📊 Resultados del modelo de regresión lineal:")
print(f"MAE (Error Absoluto Medio): {mae:.2f}")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse:.2f}")
print(f"R² (Coeficiente de determinación): {r2:.2f}")

**SEGUNDO MODELO: RANDOM FOREST REGRESSOR**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
# Crear modelo RandomForest
rf_model = RandomForestRegressor(
    n_estimators=100,   # cantidad de árboles
    max_depth= 10,       # profundidad máxima de cada árbol

    random_state=42
)
# Entrenar
rf_model.fit(X_train, y_train)
# Predecir
y_pred = rf_model.predict(X_test)

# Evaluar desempeño
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📊 Resultados RandomForestRegressor:")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.2f}")

***CURVAS DE MODELOS DE REGRESIÓN***

In [ ]:
from sklearn.model_selection import learning_curve

In [ ]:
# CURVA: Predicción vs Valor Real
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.6, color='blue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', lw=2)
plt.xlabel('Valores reales')
plt.ylabel('Predicciones')
plt.title('Predicción vs Valor real')
plt.grid(True)
plt.show()

# CURVA: Residuos
residuos = y_test - y_pred
plt.figure(figsize=(6,4))
plt.scatter(y_pred, residuos, alpha=0.6, color='purple')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Error (residuo)')
plt.title('Curva de residuos')
plt.grid(True)
plt.show()

#  CURVA: Learning Curve (Curva de aprendizaje)
train_sizes, train_scores, test_scores = learning_curve(
    rf_model, X, y, cv=5,
    scoring='r2', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(7,5))
plt.plot(train_sizes, train_mean, 'o-', label='Entrenamiento')
plt.plot(train_sizes, test_mean, 's-', label='Validación')
plt.xlabel('Tamaño del conjunto de entrenamiento')
plt.ylabel('R²')
plt.title('Curva de aprendizaje')
plt.legend()
plt.grid(True)
plt.show()

# CURVA: Importancia de características
if hasattr(rf_model, 'feature_importances_'):
    importancia = rf_model.feature_importances_
    features = [f'Var{i+1}' for i in range(X.shape[1])]
    plt.figure(figsize=(6,4))
    plt.barh(features, importancia, color='teal')
    plt.xlabel('Importancia')
    plt.ylabel('Variables')
    plt.title('Importancia de características')
    plt.grid(True, axis='x')
    plt.show()
else:
    print("⚠️ Este modelo no tiene atributo 'feature_importances_'.")

**XGBOOST REGRESSOR**

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
#dividir datos de prueba y entrenamiento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#Crear modelo XGBRegressor
modelo_xgb = XGBRegressor(
    n_estimators=200,      # número de árboles
    learning_rate=0.1,     # tasa de aprendizaje
    max_depth=6,           # profundidad de los árboles
    random_state=42
)
#Entrenar
modelo_xgb.fit(X_train, y_train)
#Predecir
y_pred = modelo_xgb.predict(X_test)
print("R²:", r2_score(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred)) #squared=False))
print("MAE:", mean_absolute_error(y_test, y_pred))

***CURVAS DE MODELOS DE REGRESIÓN***

In [ ]:
# CURVA: Predicción vs Valor Real
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.6, color='blue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', lw=2)
plt.xlabel('Valores reales')
plt.ylabel('Predicciones')
plt.title('Predicción vs Valor real')
plt.grid(True)
plt.show()

# CURVA: Residuos
residuos = y_test - y_pred
plt.figure(figsize=(6,4))
plt.scatter(y_pred, residuos, alpha=0.6, color='purple')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Error (residuo)')
plt.title('Curva de residuos')
plt.grid(True)
plt.show()

#  CURVA: Learning Curve (Curva de aprendizaje)
train_sizes, train_scores, test_scores = learning_curve(
    modelo_xgb, X, y, cv=5,
    scoring='r2', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(7,5))
plt.plot(train_sizes, train_mean, 'o-', label='Entrenamiento')
plt.plot(train_sizes, test_mean, 's-', label='Validación')
plt.xlabel('Tamaño del conjunto de entrenamiento')
plt.ylabel('R²')
plt.title('Curva de aprendizaje')
plt.legend()
plt.grid(True)
plt.show()

# CURVA: Importancia de características
if hasattr(modelo_xgb, 'feature_importances_'):
    importancia = modelo_xgb.feature_importances_
    features = [f'Var{i+1}' for i in range(X.shape[1])]
    plt.figure(figsize=(6,4))
    plt.barh(features, importancia, color='teal')
    plt.xlabel('Importancia')
    plt.ylabel('Variables')
    plt.title('Importancia de características')
    plt.grid(True, axis='x')
    plt.show()
else:
    print("⚠️ Este modelo no tiene atributo 'feature_importances_'.")

**Gradient Boosting Regressor**

In [ ]:
#importar librerías
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Crear el modelo
gbr = GradientBoostingRegressor(
    n_estimators=200,     # número de árboles
    learning_rate=0.1,    # tasa de aprendizaje
    max_depth=5,          # profundidad máxima de cada árbol
    random_state=42
)

# Entrenar el modelo
gbr.fit(X_train, y_train)
#predicción
y_pred = gbr.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R2: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

***CURVAS DE MODELOS DE REGRESIÓN***

In [ ]:
# CURVA: Predicción vs Valor Real
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.6, color='blue')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', lw=2)
plt.xlabel('Valores reales')
plt.ylabel('Predicciones')
plt.title('Predicción vs Valor real')
plt.grid(True)
plt.show()

# CURVA: Residuos
residuos = y_test - y_pred
plt.figure(figsize=(6,4))
plt.scatter(y_pred, residuos, alpha=0.6, color='purple')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicciones')
plt.ylabel('Error (residuo)')
plt.title('Curva de residuos')
plt.grid(True)
plt.show()

#  CURVA: Learning Curve (Curva de aprendizaje)
train_sizes, train_scores, test_scores = learning_curve(
    gbr, X, y, cv=5,
    scoring='r2', n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10)
)

train_mean = train_scores.mean(axis=1)
test_mean = test_scores.mean(axis=1)

plt.figure(figsize=(7,5))
plt.plot(train_sizes, train_mean, 'o-', label='Entrenamiento')
plt.plot(train_sizes, test_mean, 's-', label='Validación')
plt.xlabel('Tamaño del conjunto de entrenamiento')
plt.ylabel('R²')
plt.title('Curva de aprendizaje')
plt.legend()
plt.grid(True)
plt.show()

# CURVA: Importancia de características
if hasattr(gbr, 'feature_importances_'):
    importancia = gbr.feature_importances_
    features = [f'Var{i+1}' for i in range(X.shape[1])]
    plt.figure(figsize=(6,4))
    plt.barh(features, importancia, color='teal')
    plt.xlabel('Importancia')
    plt.ylabel('Variables')
    plt.title('Importancia de características')
    plt.grid(True, axis='x')
    plt.show()
else:
    print("⚠️ Este modelo no tiene atributo 'feature_importances_'.")

**MLP REGRESSOR (RED NEURONAL)**

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

In [ ]:
#NORMALIZACIÓN OPCIÓN 1
#from sklearn.preprocessing import MinMaxScaler
#scaler = MinMaxScaler()
#X_scaled = scaler.fit_transform(X)


In [ ]:
#NORMALIZACIÓN OPCIÓN 2
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
#dividir datos de prueba y entrenamiento
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
#Crear modelo MLPRegressor
modelo_mlp = MLPRegressor(
    hidden_layer_sizes=(100, 50),  # 2 capas ocultas
    activation='relu',
    solver='adam',
    max_iter=1000,
    random_state=42
)

#Entrenar
modelo_mlp.fit(X_train, y_train)

#Predecir
y_pred = modelo_mlp.predict(X_test)
print("R²:", r2_score(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred, ))#squared=False))
print("MAE:", mean_absolute_error(y_test, y_pred))